In [1]:
import numpy as np
import math
from scipy.spatial import Voronoi,voronoi_plot_2d
import random as rnd
import matplotlib.pyplot as plt
from IPython.display import display, HTML,Javascript,clear_output
from svg_plot import SVGPlot
import time

In [2]:
def polygon_centroid(vertices):
    vertices = np.asarray(vertices)
    x = vertices[:, 0]
    y = vertices[:, 1]
    
    # Close the polygon (first point == last point)
    x = np.append(x, x[0])
    y = np.append(y, y[0])
    
    # Shoelace formula for signed area
    cross_product = x[:-1] * y[1:] - x[1:] * y[:-1]
    area = 0.5 * np.sum(cross_product)
    
    # Centroid coordinates
    cx = (1 / (6 * area)) * np.sum((x[:-1] + x[1:]) * cross_product)
    cy = (1 / (6 * area)) * np.sum((y[:-1] + y[1:]) * cross_product)
    
    return (cx, cy)

In [20]:
class PopulationArea:
    def __init__(self,area_id,center,vertices,colour = "rgba(255, 255, 255, 0.0)"):
        self.area_id = area_id
        self.center = center
        self.polygon = vertices
        self.colour = colour
        self.local_population = []
        self.center = polygon_centroid(self.polygon)

    def point_in_polygon(self,x,y):
        polygon = np.asarray(self.polygon)
        n = len(polygon)
        inside = False
        for i in range(n):
            x1, y1 = polygon[i]
            x2, y2 = polygon[(i + 1) % n]
            
            # Check if edge crosses horizontal ray to the right
            if ((y1 > y) != (y2 > y)) and (x < (x2 - x1) * (y - y1) / (y2 - y1) + x1):
                inside = not inside
        return inside

    def create_population(self,pop_size,population):
        for _ in range(pop_size):
            person = Denizen(self)
            person.assign_pos()
            population.append(person)

    def draw(self,svg,colour=lambda p: "rgb(0,0,0)"):
        svg.add_polygon(self.polygon,self.colour)
        for p in self.local_population:
            x = p.pos[0]
            y = p.pos[1]
            svg.add_rect(x,y,colour(p),0.04)
        svg.add_circle(self.center[0]-0.05,self.center[1]-0.05,"blue",0.1)

In [21]:
def generate_age():
    def age_probability(age):
        if age < 0:
            return 0
        
        # Peak at ~38, rise then fall
        peak = 38
        rise = (age / peak) ** 1.5
        fall = np.exp(-0.035 * (age - peak))
        
        return rise * fall
    age = -1
    while age < 0:
        candidate = np.random.uniform(0, 100)
        p = age_probability(candidate)
        max_p = age_probability(38)  # Peak probability
        
        if np.random.rand() < (p / max_p):
            return int(candidate)

In [22]:
def linear_toward_zero():
    magnitude = 1 - math.sqrt(rnd.random())
    return magnitude if rnd.random() < 0.5 else -magnitude
    

In [23]:
def age_migrate_curve(x, start_point=20, end_point=70, decay_rate=None):
    if decay_rate is None:
        decay_rate = -np.log(0.01) / (end_point - start_point)
    shifted_x = x - start_point
    prob = np.exp(-decay_rate * shifted_x)
    prob = np.where(shifted_x < 0, 0, prob)
    prob = np.clip(prob, 0, 1)
    return prob

In [24]:
class Denizen:
    def __init__(self, area):
        self.area = area
        self.pos = (0,0)
        self.age = generate_age()
        self.moved = False
    
    def assign_pos(self):
        x_vals = [point[0] for point in self.area.polygon]  # x is first element
        y_vals = [point[1] for point in self.area.polygon]  # y is second element
        
        x_min, x_max = min(x_vals), max(x_vals)
        y_min, y_max = min(y_vals), max(y_vals)
        
        while True:
            x = x_min + rnd.random() * (x_max - x_min)
            y = y_min + rnd.random() * (y_max - y_min)
            
            if self.area.point_in_polygon(x, y):
                self.pos = (float(x), float(y))
                self.area.local_population.append(self)
                return

    def migrate(self):
        if rnd.random()<age_migrate_curve(self.age):
            return True


    def __str__(self):
        return f"Vector({self.area.area_id}, {self.pos},{self.age})"

In [25]:
class Simulation:
    def __init__(self):
        self.areas = []
        self.population=[]
        
    def create_areas(self,number_of_areas):
        points = np.random.rand(number_of_areas, 2) * 10  # NUMBER_OF_AREAS rows, 2 columns, scale to 0-10
        vor = Voronoi(points)
        area_id = 0;
        for i in range(len(points)):
            region_idx = vor.point_region[i]
            region = vor.regions[region_idx]
            if region and -1 not in region: #if region` - Ensures the region is not empty -1 not in region` - Filters out unbounded regions
                vertices = vor.vertices[region]
                x_min, y_min = vertices.min(axis=0)  # [min_x, min_y]
                x_max, y_max = vertices.max(axis=0)  # [max_x, max_y]
                if(max([x_max,y_max])<10) and (min([x_min,y_min])>0):
                    self.areas.append(PopulationArea(area_id,points[i],vertices))
                    area_id +=1

    def get_area(self,x,y):
        for area in self.areas:
            if area.point_in_polygon(x,y):
                return True, area
        return False, None
                  
    def create_population(self,local_pop_size):
        for area in self.areas:
            area.create_population(local_pop_size,self.population)
            
    def create_a_population(self,local_pop_size):
        self.areas[1].create_population(local_pop_size,self.population)

    def migrate(self):
        for p in self.population:
            if (p.moved)==False and (p.migrate()):
                flag = False
                while flag == False:
                    x = linear_toward_zero()*10
                    y = linear_toward_zero()*10
                    flag, area = self.get_area(x,y)
                    if flag == True:
                        p.pos=(x,y)
                        p.area.local_population.remove(p)
                        p.area= area
                        area.local_population.append(p)
                        p.moved = True  

    def draw(self,svg):
        for area in self.areas:
            area.draw(svg,colour=lambda p: f'rgb(522,{p.age*3},{p.age*3})')


In [26]:
np.random.seed(42)  # For reproducible results
sim = Simulation()
sim.create_areas(60)
print(f"{len(sim.areas)} areas created") 
sim.create_a_population(int(np.random.normal(loc=1050, scale=20)))
print(f"A population of {len(sim.population)} created") 
svg = SVGPlot(500, 500, 10)
output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)
for t in range (100):
    sim.migrate()
    svg.clear()
    sim.draw(svg) 
    new_svg = svg.get_canvas()
    new_svg += f'<p> Iteration: {t} weeks </p>'
    output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
    time.sleep(0.1)

37 areas created
A population of 1035 created
